# P2Rank + Vina: Full Single Run Docking Example

This notebook demonstrates a complete Vina docking run using **P2Rank** for blind binding site prediction,
compared against the traditional approach using the original ligand location.

**Workflow:**
1. Set up inputs (protein PDB, ligand SMILES)
2. Run Guild **without** P2Rank (original ligand location)
3. Run Guild **with** P2Rank (predicted binding site)
4. Compare Vina scores and box locations

In [ ]:
import logging
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Enable logging so we can see what's happening
logging.basicConfig(level=logging.INFO, format="%(name)s - %(levelname)s - %(message)s")

print(f"Project root: {project_root}")

## 1. Define inputs

Using `8gut` protein (GPCR) with chain R, original ligand KO8.

In [ ]:
# Protein setup
protein_file = os.path.join(project_root, "notebooks/data/pdbs/8gut.pdb")
protein_idx = "8gut"
protein_chain = "R"
original_ligand = "KO8"
original_ligand_chain = "R"

# Ligand setup (a natural product from the existing dataset)
ligand_smile = "CC(=CCC1=CC(=O)C=CC1=O)CCc1c(C)cc(O)c(C=O)c1C"
ligand_idx = "CNP0294578"

print(f"Protein: {protein_idx} (chain {protein_chain})")
print(f"Original ligand: {original_ligand}")
print(f"Ligand SMILES: {ligand_smile}")
print(f"Protein file exists: {os.path.exists(protein_file)}")

## 2. Traditional run — original ligand location

In [ ]:
from guild.docking.vina import get_center_and_size_from_box_file
from guild.run import Guild

wizard_traditional = Guild(
    ligand_smile=ligand_smile,
    ligand_idx=ligand_idx,
    protein_idx=protein_idx,
    protein_file=protein_file,
    project_name="p2rank_demo_traditional",
    protein_chain=protein_chain,
    original_ligand=original_ligand,
    original_ligand_chain=original_ligand_chain,
    use_gpu=False,
    predict_binding_pocket=False,  # Use original ligand location
)

print("Guild initialized (traditional mode)")

In [ ]:
# Check the Vina box generated from original ligand location
traditional_center, traditional_size = get_center_and_size_from_box_file(wizard_traditional.vina_box)
print(f"Traditional box center: {traditional_center}")
print(f"Traditional box size:   {traditional_size}")

In [ ]:
# Run Vina docking with traditional box
report_trad = wizard_traditional.run_autodock_vina()
print(f"\nVina completed with status: {'success' if report_trad == 0 else 'failed'}")

In [ ]:
# Read traditional scores
with open(wizard_traditional.vina_output_scores, 'r') as f:
    traditional_scores = f.read()
print("Traditional Vina scores (kcal/mol):")
print(traditional_scores)

## 3. P2Rank run — predicted binding site

In [ ]:
wizard_p2rank = Guild(
    ligand_smile=ligand_smile,
    ligand_idx=ligand_idx,
    protein_idx=protein_idx,
    protein_file=protein_file,
    project_name="p2rank_demo_predicted",
    protein_chain=protein_chain,
    use_gpu=False,
    predict_binding_pocket=True,  # Use P2Rank binding site prediction
)

print("Guild initialized (P2Rank mode)")

In [ ]:
# Check the Vina box generated from P2Rank prediction
p2rank_center, p2rank_size = get_center_and_size_from_box_file(wizard_p2rank.vina_box)
print(f"P2Rank box center: {p2rank_center}")
print(f"P2Rank box size:   {p2rank_size}")

In [ ]:
# Run Vina docking with P2Rank-predicted box
report_p2rank = wizard_p2rank.run_autodock_vina()
print(f"\nVina completed with status: {'success' if report_p2rank == 0 else 'failed'}")

In [ ]:
# Read P2Rank scores
with open(wizard_p2rank.vina_output_scores, 'r') as f:
    p2rank_scores = f.read()
print("P2Rank Vina scores (kcal/mol):")
print(p2rank_scores)

## 4. Compare results

In [ ]:
import numpy as np

# Compare box centers
distance = np.linalg.norm(np.array(traditional_center) - np.array(p2rank_center))

print("=" * 60)
print("BOX CENTER COMPARISON")
print("=" * 60)
print(f"Traditional (ligand-based):  {traditional_center}")
print(f"P2Rank (predicted):          {p2rank_center}")
print(f"Distance between centers:    {distance:.3f} Angstrom")
print()
print(f"Traditional box size: {traditional_size}")
print(f"P2Rank box size:      {p2rank_size}")
print()
print("=" * 60)
print("VINA SCORES COMPARISON (best pose, kcal/mol)")
print("=" * 60)

# Parse best scores
trad_best = float(traditional_scores.strip().split('\n')[0].split(':')[1].strip())
p2r_best = float(p2rank_scores.strip().split('\n')[0].split(':')[1].strip())

print(f"Traditional best score: {trad_best:.2f} kcal/mol")
print(f"P2Rank best score:      {p2r_best:.2f} kcal/mol")
print(f"Difference:             {abs(trad_best - p2r_best):.2f} kcal/mol")

In [ ]:
# Clean up test projects (optional - uncomment to remove)
# import shutil
# shutil.rmtree(wizard_traditional.project_dir, ignore_errors=True)
# shutil.rmtree(wizard_p2rank.project_dir, ignore_errors=True)
# print("Cleaned up test projects.")